# 02 FRED Ingest and Clean

Fetch required national FRED series using controlled API calls, preserve raw frequencies, annualize transparently, and build the national housing/income processed dataset.

## Annualization and Inflation Rules

Raw observations are preserved in `data/raw/fred/`. Weekly, monthly, and quarterly data are annualized using annual averages. `CPIAUCSL` provides the latest complete-year inflation base. `MEHOINUSA672N` is already real and is not deflated again.

In [ ]:
from pathlib import Path
import json, os, requests
import pandas as pd
import numpy as np
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
DATA_DIR=ROOT/'data'; RAW_DIR=DATA_DIR/'raw'; INTERIM_DIR=DATA_DIR/'interim'; PROCESSED_DIR=DATA_DIR/'processed'; REPORTS_DIR=ROOT/'reports'
for p in [RAW_DIR/'fred', INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR]: p.mkdir(parents=True, exist_ok=True)
def load_local_env():
    env=ROOT/'.env'
    if env.exists():
        for line in env.read_text().splitlines():
            if line.strip() and not line.strip().startswith('#') and '=' in line:
                k,v=line.split('=',1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
try:
    from dotenv import load_dotenv; load_dotenv(ROOT/'.env')
except Exception: load_local_env()
def get_api_key(name, required=False):
    value=os.environ.get(name)
    if value: return value
    sp=ROOT/'.secrets'/'api_keys.json'
    if sp.exists():
        try:
            value=json.loads(sp.read_text()).get(name)
            if value: os.environ.setdefault(name,value); return value
        except json.JSONDecodeError: pass
    if required: raise RuntimeError(f'Missing {name}. Set it as an environment variable or in local .env.')
    return None
def write_csv(df,path):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True); df.to_csv(path,index=False); print(f'wrote {path.relative_to(ROOT)} ({len(df):,} rows)')

In [ ]:
FRED_BASE='https://api.stlouisfed.org/fred'; FRED_API_KEY=get_api_key('FRED_API_KEY', required=True)
series_plan={'MSPUS':{'role':'median_home_price_nominal','required':True},'CPIAUCSL':{'role':'cpi_all_items','required':True},'MEHOINUSA672N':{'role':'real_median_household_income','required':True},'MORTGAGE30US':{'role':'mortgage_rate_30y','required':True},'CSUSHPINSA':{'role':'case_shiller_index','required':True},'CUSR0000SAH1':{'role':'shelter_cpi','required':False},'CUSR0000SAS4':{'role':'transportation_services_cpi','required':False}}
def fred_observations(series_id):
    r=requests.get(f'{FRED_BASE}/series/observations', params={'series_id':series_id,'api_key':FRED_API_KEY,'file_type':'json'}, timeout=60)
    if r.status_code!=200: raise RuntimeError(f'FRED observations failed for {series_id}: HTTP {r.status_code} {r.text[:160]}')
    df=pd.DataFrame(r.json().get('observations',[]))
    if df.empty: raise RuntimeError(f'No observations returned for {series_id}')
    df['series_id']=series_id; df['date']=pd.to_datetime(df['date']); df['value']=pd.to_numeric(df['value'].replace('.',np.nan), errors='coerce')
    return df[['series_id','date','value','realtime_start','realtime_end']]
raw_frames=[]; meta=[]
for sid,plan in series_plan.items():
    try:
        df=fred_observations(sid); write_csv(df, RAW_DIR/'fred'/f'{sid}.csv'); raw_frames.append(df)
        meta.append({'series_id':sid,'role':plan['role'],'required':plan['required'],'status':'loaded','observations':len(df),'first_date':df.date.min().date().isoformat(),'last_date':df.date.max().date().isoformat(),'missing_values':int(df.value.isna().sum())})
    except Exception as exc:
        meta.append({'series_id':sid,'role':plan['role'],'required':plan['required'],'status':'failed','observations':0,'first_date':'','last_date':'','missing_values':'','error':str(exc)})
        if plan['required']: raise
fred_raw=pd.concat(raw_frames, ignore_index=True)
write_csv(pd.DataFrame(meta), INTERIM_DIR/'fred_validation_summary.csv')
fred_raw['year']=fred_raw.date.dt.year
annual=fred_raw.groupby(['series_id','year'], as_index=False).agg(value=('value','mean'), observations=('value','count'), first_date=('date','min'), last_date=('date','max'))
write_csv(annual, INTERIM_DIR/'fred_annualized.csv')
wide=annual.pivot(index='year', columns='series_id', values='value').reset_index(); wide.columns.name=None
complete=annual[(annual.series_id=='CPIAUCSL') & (annual.observations>=12)]['year']
base_year=int(complete.max()); cpi_base=float(wide.loc[wide.year==base_year,'CPIAUCSL'].iloc[0])
processed=pd.DataFrame({'year':wide.year})
processed['cpi_index']=wide.get('CPIAUCSL'); processed['cpi_base_year']=base_year
processed['median_home_price_nominal']=wide.get('MSPUS')
processed['real_median_home_price']=processed.median_home_price_nominal*(cpi_base/processed.cpi_index)
processed['real_median_household_income']=wide.get('MEHOINUSA672N')
processed['home_price_to_real_income_ratio']=processed.real_median_home_price/processed.real_median_household_income
processed['median_home_price_to_median_household_income_ratio']=processed.home_price_to_real_income_ratio
processed['mortgage_rate_annual_avg']=wide.get('MORTGAGE30US'); processed['case_shiller_index']=wide.get('CSUSHPINSA')
first_home=processed.real_median_home_price.dropna().iloc[0]; processed['real_home_price_index']=processed.real_median_home_price/first_home*100
if processed.case_shiller_index.notna().any():
    first_case=processed.case_shiller_index.dropna().iloc[0]
    processed['case_shiller_index_first_available']=processed.case_shiller_index/first_case*100
    processed['case_shiller_relative_to_cpi']=processed.case_shiller_index/processed.cpi_index*100
for source_col,out_col in {'CUSR0000SAH1':'shelter_cpi','CUSR0000SAS4':'transportation_services_cpi'}.items():
    if source_col in wide.columns: processed[out_col]=wide[source_col]
validation=[]
for col in processed.columns:
    if col=='year': continue
    s=processed[col]
    validation.append({'metric':col,'non_null_years':int(s.notna().sum()),'missing_years':int(s.isna().sum()),'first_valid_year':int(processed.loc[s.notna(),'year'].min()) if s.notna().any() else None,'last_valid_year':int(processed.loc[s.notna(),'year'].max()) if s.notna().any() else None,'min_value':float(s.min()) if s.notna().any() else None,'max_value':float(s.max()) if s.notna().any() else None})
write_csv(pd.DataFrame(validation), INTERIM_DIR/'fred_processed_validation.csv')
write_csv(processed, PROCESSED_DIR/'national_housing_income.csv')
processed.tail()